Tensorflow and Keras were installed following the official tutorial:
https://www.tensorflow.org/install/pip

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sys
sys.path.append("..")
import methods.explainability as ex
from methods import utils
from methods.chromagenet import CalibratedModel, make_model

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from betacal import BetaCalibration

physical_devices = tf.config.list_physical_devices("GPU")
print("Num GPUs:", len(physical_devices))

In [ ]:
pal = utils.get_class_palette()
mic_pal = utils.get_microscopist_palette()

In [ ]:
NORM = "none"
RES = 0.05
k_cv = 5
IMGS_DIR = f"../../data/preprocessed/2D_res={RES}_norm={NORM}_{k_cv}fold_withDMSO"

CNN_DENSE_FILTS = (256, 64, 16)
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 64
CHANNEL_MODE = "grayscale"
N_OUTPUT_UNITS = 1
LR_SCHED = None
OPTIMIZER = "AdamW"
COMMENT = "testing baseline"

EPOCHS = 50
LR = 1e-4
SEED = 2023

if CHANNEL_MODE == "rgb":
    channels = (3,)
elif CHANNEL_MODE == "grayscale":
    channels = (1,)

if N_OUTPUT_UNITS == 1:
    OUTPUT_FUNC = "sigmoid"
    LOSS_FUNC = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=0.1,
    )
    LABEL_MODE = "binary"
elif N_OUTPUT_UNITS == 2:
    OUTPUT_FUNC = "softmax"
    LOSS_FUNC = "categorical_crossentropy"
    LABEL_MODE = "categorical"

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(factor=0.5, fill_mode="constant"),
        layers.RandomContrast(factor=0.05),
    ]
)

In [ ]:
n_0 = 0
n_1 = 1

for k in range(1, k_cv + 1):
    train_ds = None
    print(f"Starting fold {k}")
    ds = tf.keras.utils.image_dataset_from_directory(
        f"{IMGS_DIR}/fold_{k}/",
        color_mode=CHANNEL_MODE,
        labels="inferred",
        label_mode=LABEL_MODE,
        interpolation="bilinear",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    
    for _, y in ds:
        unique, counts = np.unique(y.numpy(), return_counts=True)
        count_dict = dict(zip(unique, counts))
        
        n_0 += count_dict.get(0, 0)  # Add count of class 0
        n_1 += count_dict.get(1, 0)  # Add count of class 1

    print(f"Fold {k}: Class 0 = {n_0}, Class 1 = {n_1}")

# Final total counts
print(f"Total images: Class 0 = {n_0}, Class 1 = {n_1}")

In [ ]:
testing_ds = tf.keras.utils.image_dataset_from_directory(
        f"{IMGS_DIR}/fold_1/",
        color_mode=CHANNEL_MODE,
        labels="inferred",
        label_mode=LABEL_MODE,
        interpolation="bilinear",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in testing_ds.take(1):

    young_images = images[labels.numpy().ravel() == 1.]
    
    for i in range(9):
        print(np.max(young_images[i]))
        print(np.min(young_images[i]))
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(young_images[i].numpy().astype("uint8"))
        plt.axis("off")
    plt.tight_layout()

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in testing_ds.take(1):

    aged_images = images[labels.numpy().ravel() == 0.]
    
    for i in range(9):
        print(np.max(aged_images[i]))
        print(np.min(aged_images[i]))
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(aged_images[i].numpy().astype("uint8"))
        plt.axis("off")
    plt.tight_layout()

In [ ]:
# Apply `data_augmentation` to the training images.
testing_ds = testing_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf.data.AUTOTUNE,
)

plt.figure(figsize=(10, 10))
for images, labels in testing_ds.take(1):
    for i in range(9):
        print(np.max(images[i]))
        print(np.min(images[i]))
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(int(labels[i]))
        plt.axis("off")

In [ ]:
# Build ChromAgeNet model
model = make_model(
    input_shape=IMAGE_SIZE + channels,
    dense_units=CNN_DENSE_FILTS,
    n_output_units=N_OUTPUT_UNITS,
    output_activation=OUTPUT_FUNC,
)

In [ ]:
model.summary()

In [ ]:
# keras.utils.plot_model(model, show_shapes=True)

In [ ]:
os.chdir(f"../../../notebooks")

In [ ]:
import os
for k in range(1, k_cv + 1):
    os.makedirs(f"../models/checkpoints/fold_{k}", exist_ok=True)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "{epoch}-{val_accuracy:.3f}.weights.h5",
        monitor="val_auc",
        save_weights_only=True,
        save_best_only=False,
    )
]


lr_obj = None
if LR_SCHED == "cosine_decay":
    lr_obj = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR, decay_steps=100, alpha=1e-1
    )

if lr_obj is None:
    lr_obj = LR

if OPTIMIZER == "AdamW":
    optim = keras.optimizers.AdamW(
        learning_rate=lr_obj, use_ema=True
    )
elif OPTIMIZER == "Adam":
    optim = keras.optimizers.Adam(learning_rate=lr_obj, use_ema=True)
elif OPTIMIZER == "SGD":
    optim = keras.optimizers.SGD(learning_rate=lr_obj, momentum=0.6)

metrics = [
    "accuracy",
    tf.keras.metrics.AUC(),
    tf.keras.metrics.Precision(),
    tf.keras.metrics.Recall(),
    tf.keras.metrics.F1Score(),
]

def plot_loss(hist):
    plt.plot(hist["loss"])
    plt.plot(hist["val_loss"])
    plt.title("Training Progress")
    plt.ylabel("Loss")
    plt.xlabel("Epochs")
    plt.legend(["train_loss", "val_loss"], loc="upper left")
    plt.show()


def plot_acc(hist):
    plt.plot(hist["accuracy"])
    plt.plot(hist["val_accuracy"])
    plt.title("Training Progress")
    plt.ylabel("Accuracy")
    plt.xlabel("Epochs")
    plt.legend(["train_acc", "val_acc"], loc="upper left")
    plt.show()


In [ ]:
sns.set_style("whitegrid")
train_ds = None
IMGS_DIR = f"../../../../data/preprocessed/2D_res={RES}_norm={NORM}_{k_cv}fold_withDMSO"

model.compile(
    optimizer=optim,
    loss=LOSS_FUNC,
    metrics=metrics,
    jit_compile=True,
)

initial_weights = model.get_weights()

for k in range(1, k_cv + 1):
    train_ds = None

    os.chdir(f"../models/checkpoints/fold_{k}")
    print("Saving model weights at: ", os.getcwd())
    
    print(f"Starting fold {k}")
    for i in range(1, k_cv + 1):
        if i == k:
            val_ds = tf.keras.utils.image_dataset_from_directory(
                f"{IMGS_DIR}/fold_{i}/",
                color_mode=CHANNEL_MODE,
                labels="inferred",
                label_mode=LABEL_MODE,
                interpolation="bilinear",
                seed=SEED,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                shuffle=False,
            )
            continue

        if train_ds:
            train_ds = train_ds.concatenate(
                tf.keras.utils.image_dataset_from_directory(
                    f"{IMGS_DIR}/fold_{i}/",
                    color_mode=CHANNEL_MODE,
                    labels="inferred",
                    label_mode=LABEL_MODE,
                    interpolation="bilinear",
                    seed=SEED,
                    image_size=IMAGE_SIZE,
                    batch_size=BATCH_SIZE,
                    shuffle=True,
                )
            )
        else:
            train_ds = tf.keras.utils.image_dataset_from_directory(
                f"{IMGS_DIR}/fold_{i}/",
                color_mode=CHANNEL_MODE,
                labels="inferred",
                label_mode=LABEL_MODE,
                interpolation="bilinear",
                seed=SEED,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                shuffle=True,
            )

    n_train_ims = train_ds.cardinality().numpy() * BATCH_SIZE
    print(n_train_ims)

    train_ds = train_ds.unbatch().shuffle(10000).batch(BATCH_SIZE)
    # Prefetching samples in GPU memory helps maximize GPU utilization.
    train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

    train_ds = train_ds.map(
        lambda img, label: (data_augmentation(img), label),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    model.set_weights(initial_weights)

    history = model.fit(
        train_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        validation_data=val_ds,
        shuffle=True,
        sample_weight=None,
    )

    hist = utils.fix_repeated_metric_names(model.history.history)
    hist_df = ex.get_history_df(hist)
    ex.plot_training(hist_df)
    plot_loss(model.history.history)
    plot_acc(model.history.history)

    model.evaluate(val_ds)

    os.chdir(f"../../../notebooks")
    model.save(f"../models/res{RES}_norm{NORM}_fold{k}.keras")


In [ ]:
best_fold_models = [
    "../models/checkpoints/fold_1/44-0.692.weights.h5",
    "../models/checkpoints/fold_2/48-0.639.weights.h5",
    "../models/checkpoints/fold_3/50-0.678.weights.h5",
    "../models/checkpoints/fold_4/43-0.658.weights.h5",
    "../models/checkpoints/fold_5/42-0.651.weights.h5",
]

def find_optimal_t(tpr, fpr, thresholds):
    # Find optimal threshold using Youden’s J statistic
    J_scores = tpr - fpr
    best_idx = np.argmax(J_scores)
    return thresholds[best_idx]


def plot_class_distplot(df, x, hue, threshold, palette):
    sns.displot(df, x=x, hue=hue, binwidth=0.04, 
                palette=pal, height=3, kde=True)
    # Add a vertical line at the best threshold
    plt.axvline(threshold, color='red', linestyle='--', linewidth=2)
    plt.xlabel("Averaged probability (whole nucleus)")
    plt.ylabel("Number of nuclei")
    plt.xlim(0, 1)
    plt.show()


def compute_ece(y_true, y_probs, n_bins=10):

    bins = np.linspace(0, 1, n_bins + 1)  # Create bin edges
    bin_indices = np.digitize(y_probs, bins) - 1  # Assign samples to bins
    ece = 0.0

    for i in range(n_bins):
        mask = bin_indices == i
        if np.sum(mask) > 0:
            acc = np.mean(y_true[mask])  # Fraction of positives in bin
            conf = np.mean(y_probs[mask])  # Mean predicted probability in bin
            ece += np.abs(acc - conf) * (np.sum(mask) / len(y_probs))  # Weighted average

    return ece


In [ ]:
import sklearn.metrics as skmetrics

aucs_df = pd.DataFrame(columns=["TPR", "FPR", "fold"])
pr_df = pd.DataFrame(columns=["Precision", "Recall", "fold"])
conf_2D_df = pd.DataFrame(columns=["Confusion", "index", "fold"])
conf_3D_df = pd.DataFrame(columns=["Confusion", "index", "fold"])
cv_3D_df = pd.DataFrame()
cv_2D_df = pd.DataFrame()

eces, cal_eces = [], []
briers, cal_briers = [], []

IMGS_DIR = f"../../data/preprocessed/2D_res={RES}_norm={NORM}_{k_cv}fold_withDMSO"

for fold in range(1, k_cv + 1):
    train_ds = None
    threshold_list = []
    print(f"Starting fold {fold}")

    model.load_weights(best_fold_models[fold-1])
    
    val_ids, val_ds = utils.dataset_from_partition(
        IMGS_DIR,
        f"fold_{fold}",
        CHANNEL_MODE,
        LABEL_MODE,
        IMAGE_SIZE,
        BATCH_SIZE,
        SEED,
    )
    val_df = ex.get_output_df_voting(val_ids, val_ds, model, LABEL_MODE)
    val_df["class"] = val_df["label"].map({0: 'aged', 1: 'young'})
    val_df["fold"] = fold
    y = np.concatenate([y for _, y in val_ds], axis=0)

    print(val_ds.cardinality().numpy() * BATCH_SIZE)
    y_prob = model.predict(val_ds)

    # Compute ROC curve
    fpr, tpr, thresholds = skmetrics.roc_curve(y, y_prob)
    aucs_df = pd.concat([aucs_df, pd.DataFrame(
        {"TPR":tpr, "FPR":fpr, "fold":fold})]
    )
    best_t = find_optimal_t(tpr, fpr, thresholds)
    threshold_list.append(best_t)

    # Compute PR curve
    precision, recall, thresholds = skmetrics.precision_recall_curve(y, y_prob)
    pr_df = pd.concat([pr_df, pd.DataFrame(
        {"Precision":precision, "Recall":recall, "fold":fold})]
    )

    # Compute confusion matrices
    y_pred = (y_prob > best_t).astype(int)
    conf_mat_2D = skmetrics.confusion_matrix(y, y_pred, normalize="true").flatten()
    conf_2D_df = pd.concat([conf_2D_df, pd.DataFrame(
        {"Confusion":conf_mat_2D, "index":[1, 2, 3, 4], "fold":fold})]
    )

    cv_2D_df = pd.concat([cv_2D_df, 
                          pd.DataFrame(
                              {"nuc_id": np.concatenate([y for y in val_ids], axis=0), 
                               "label": y.ravel(),
                               "pred": y_pred.ravel(),
                               "y_prob": y_prob.ravel(),
                               "fold": fold,
                              })]
                        )

    val_df["soft_pred"] = (val_df["mean_prob"] >= best_t).astype(int)
    conf_mat_3D = skmetrics.confusion_matrix(val_df["label"], val_df["soft_pred"], normalize="true").flatten()
    conf_3D_df = pd.concat([conf_3D_df, pd.DataFrame(
        {"Confusion":conf_mat_3D, "index":[1, 2, 3, 4], "fold":fold})]
    )
    cv_3D_df = pd.concat([cv_3D_df, val_df])

    ex.plot_auc(val_df["label"], val_df["mean_prob"], threshold=best_t)
    ex.plot_conf_mat(val_df["label"], val_df["mean_prob"], threshold=best_t)
    plot_class_distplot(val_df, x="mean_prob", hue="class", threshold=best_t, palette=pal)

    ir = IsotonicRegression(out_of_bounds = 'clip')
    lr = LogisticRegression(solver='lbfgs')
    bc = BetaCalibration(parameters="abm")
    #calibrators = [ir, lr, bc]
    calibrators = [bc]
    
    uncal_probs = model.predict(val_ds)
    test_labels = np.concatenate([y for _, y in val_ds], axis=0)
    
    cal_error = compute_ece(test_labels, uncal_probs, n_bins=10)
    bs = brier_score_loss(test_labels, uncal_probs)
    
    print("Pre-Calibration Error:", cal_error)
    print("Pre-Calibration Brier Score:", bs)
    eces.append(cal_error)
    briers.append(bs)
    
    nuc_prob_outputs = [val_df["mean_prob"]]
    plot_probs = [uncal_probs]
    
    for calibrator in calibrators:

        print(f"Calibrating with {calibrator}")
    
        calibrator.fit(uncal_probs.ravel(), test_labels.ravel())
        cal_probs = calibrator.predict(uncal_probs)
        
        cal_error = compute_ece(test_labels, cal_probs, n_bins=10)
        bs = brier_score_loss(test_labels, cal_probs)
        
        print("Post-Calibration Error:", cal_error)
        print("Post-Calibration Brier Score:", bs)

        cal_eces.append(cal_error)
        cal_briers.append(bs)
    
        cal_val_df = ex.get_output_df_voting(val_ids, val_ds, model, 
                                             LABEL_MODE, thresh=0.5, 
                                             cal_model=calibrator)
        cal_val_df["class"] = cal_val_df["label"].map({0: 'aged', 1: 'young'})
        nuc_prob_outputs.append(cal_val_df["mean_prob"])
        plot_probs.append(cal_probs)

        # Compute ROC curve
        cal_fpr, cal_tpr, cal_thresholds = skmetrics.roc_curve(y, cal_probs)
        best_cal_t = find_optimal_t(cal_tpr, cal_fpr, cal_thresholds)
        threshold_list.append(best_cal_t)
        ex.plot_conf_mat(cal_val_df["label"], cal_val_df["mean_prob"], threshold=best_cal_t)
        plot_class_distplot(cal_val_df, x="mean_prob", 
                            hue="class", threshold=best_cal_t, palette=pal)

    legends = ["Uncalibrated", "Beta Calib."] # "Isotonic Reg.", "Platt Scaling",
    labels = [cal_val_df["label"], cal_val_df["label"], cal_val_df["label"]]
    utils.plot_multiple_auc(labels, nuc_prob_outputs, legends, threshold_list)

    # 5. Create the integrated model
    calibrated_model = CalibratedModel(model, calibrator)
    
    # 6. Save the integrated model
    save_path = f"../models/chromagenet/calmodel_fold_{fold}"
    calibrated_model.save(save_path)

    utils.plot_calibration_curve(test_labels, plot_probs, legends)

In [ ]:
print(np.mean(eces), np.std(eces))
print(np.mean(cal_eces), np.std(cal_eces))
print(np.mean(briers), np.std(briers))
print(np.mean(cal_briers), np.std(cal_briers))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def build_performance_table(df, pred_col="soft_pred", prob_col="mean_prob"):
    
    # Assuming your DataFrame is named df
    metrics_per_fold = {}
    
    for fold, data in df.groupby("fold"):
        y_true = data["label"]  # Ground truth labels
        y_pred = data[pred_col]  # Model predictions (binary)
        y_scores = data[prob_col]  # Probability scores
    
        metrics_per_fold[fold] = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred),
            "AUC": roc_auc_score(y_true, y_scores),
            "F1": f1_score(y_true, y_pred),
        }
    
    # Convert results to a DataFrame
    metrics_df = pd.DataFrame.from_dict(metrics_per_fold, orient="index")
    
    # Compute mean, std, and 95% CI
    sum_df = metrics_df.agg(["mean", "std"])
    
    z_score = 1.96
    ci_lower = sum_df.loc["mean"] - (z_score * sum_df.loc["std"] / np.sqrt(k_cv))
    ci_upper = sum_df.loc["mean"] + (z_score * sum_df.loc["std"] / np.sqrt(k_cv))
    
    sum_df.loc["CI"] = [(round(low, 3), round(up, 3)) for low, up in zip(ci_lower, ci_upper)]
    metrics_df = pd.concat([metrics_df, sum_df])
    return metrics_df

In [ ]:
sum_2d_df = build_performance_table(cv_2D_df, pred_col="pred", prob_col="y_prob")
sum_2d_df

In [ ]:
sum_2d_df.to_latex(float_format="%.3f")

In [ ]:
soft_df = build_performance_table(cv_3D_df, pred_col="soft_pred")
soft_df

In [ ]:
hard_df = build_performance_table(cv_3D_df, pred_col="hard_preds")
hard_df

In [ ]:
soft_df.to_latex(float_format="%.3f")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, auc

sns.set_theme(style="whitegrid")
plt.figure(figsize=(5, 4))

optimal_thresholds = {}  # Store optimal thresholds per fold

for fold, group in cv_df.groupby("fold"):
    print(f"Processing fold {fold}")
    y_true = group["label"]  # True labels
    y_prob = group["mean_prob"]  # Soft voting probabilities
    
    # Compute Precision-Recall curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    
    # Compute F1-score for each threshold
    f1_scores = (2 * precision * recall) / (precision + recall + 1e-6)  # Avoid division by zero
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]
    print(f"Optimal threshold for fold {fold}: {best_threshold:.3f}")
    
    optimal_thresholds[fold] = best_threshold  # Store best threshold
    
    # Compute AUC-PR for this fold
    auc_pr = average_precision_score(y_true, y_prob)
    
    # Plot Precision-Recall curve using Seaborn
    sns.lineplot(x=recall, y=precision, label=f'Fold {fold} (AUC-PR = {auc_pr:.2f})')

    # Mark the optimal threshold point without adding it to the legend
    sns.scatterplot(x=[recall[best_idx]], y=[precision[best_idx]], color='red', s=60, zorder=3)

# Formatting
sns.despine()
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Precision-Recall Curves per Fold", fontsize=14)
plt.legend(loc="best", fontsize=9)
plt.show()

# Print optimal thresholds per fold
print("Optimal thresholds per fold:", optimal_thresholds)

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(5, 4))

for fold, group in cv_df.groupby("fold"):
    y_true = group["label"]  # True labels
    y_prob = group["mean_prob"]  # Soft voting probabilities
    
    # Compute ROC curve and AUC
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    # Find optimal threshold using Youden’s J statistic
    J_scores = tpr - fpr
    best_idx = np.argmax(J_scores)
    best_threshold = thresholds[best_idx]
    
    # Plot per fold
    sns.lineplot(x=fpr, y=tpr, label=f'Fold {fold} (AUC = {roc_auc:.2f})')
    sns.scatterplot(x=[fpr[best_idx]], y=[tpr[best_idx]], color='red', s=60, zorder=3, label="_nolegend_")


# Plot chance line
plt.plot([0, 1], [0, 1], 'k--', label="Random Guessing (AUC = 0.50)")

# Formatting
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curves per Fold")
plt.legend(loc="best", fontsize=9)

# Show plot
plt.show()

In [ ]:
def plot_conf_mat(conf_df, title="Confusion matrix"):
    
    summ_df = conf_df.groupby("index")["Confusion"].agg(["mean", "std"]).reset_index()
    conf_plot = summ_df["mean"].to_numpy().reshape((2, 2))
    
    annot = np.array([
        f"{m:.3f}% \n ± {s:.3f}" for m, s in zip(summ_df["mean"], summ_df["std"])
    ]).reshape((2, 2))
    
    plt.figure(figsize=(4, 3))
    sns.heatmap(conf_plot, annot=annot, fmt='', cmap="Blues")
    plt.xticks([0.5, 1.5], ["Aged", "Young"])
    plt.yticks([0.5, 1.5], ["Aged", "Young"])
    plt.title(title)
    plt.show()

plot_conf_mat(conf_2D_df, title="Confusion matrix (2D XY slide)")

In [ ]:
plot_conf_mat(conf_3D_df, title="Confusion matrix (whole 3D nucleus)")

In [ ]:
plt.figure(figsize=(4, 3))
sns.lineplot(data=aucs_df, x="FPR", y="TPR", hue="fold")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random (AUC=0.5)")
plt.show()

In [ ]:
plt.figure(figsize=(4, 3))
sns.lineplot(data=pr_df, x="Recall", y="Precision", hue="fold")
plt.ylim((0.4, 1))
baseline = sum(y) / len(y)
plt.axhline(y=baseline, color="gray", linestyle="--", label="Random Baseline")
plt.show()

In [ ]:
train_df = ex.get_output_df_voting(train_ids, train_ds, model, LABEL_MODE)
val_df = ex.get_output_df_voting(val_ids, val_ds, model, LABEL_MODE)

In [ ]:
dataset_names = ["train", "val"]
dataframes = [train_df, val_df]
thresholds = np.arange(0.3, 0.8, 0.05)

thresh_df = ex.plot_metrics(dataframes, dataset_names, thresholds)

for m_name in ["acc", "precision", "recall", "f1"]:
    sns.lineplot(x="threshold", y=m_name, hue="dataset", data=thresh_df)
    plt.title(m_name)
    plt.show()